In [ ]:
# Imports 
import csv
from pathlib import Path
import pandas as pd
import numpy as np
import os
from scipy.stats import spearmanr, ttest_rel, f_oneway, linregress
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
import math
import pingouin as pg
import random
from datetime import datetime
from scipy.signal import resample
from scipy.interpolate import interp1d
import matplotlib as mpl

In [ ]:
# Gets all day directories (like D01, D02, etc.) for a specific subject.
# Filters folder names that begin with "D", sorts them, and returns the list of paths.
def get_day_paths(subject_path: Path): 
    days = os.listdir(subject_path)
    day_paths = sorted(list(map(lambda x: subject_path / x, list(filter(lambda x: x[0] == "D", days)))))

    return day_paths

In [ ]:
# Reads pre-trimmed CSV files for each sensor (left/right finger, left/right wrist) for a single day or a single segment of a day
# Returns a dictionary of DataFrames with keys: "left finger", "right finger", "left wrist", "right wrist"
def read_trimmed_sensor_csv(day_path: Path): # checked
    """
    Read all sensor data. Concatenate and order if there are more than one file.
    """
    left_finger_path: Path = list(day_path.glob("left finger.csv"))[0]
    right_finger_path: Path = list(day_path.glob("right finger.csv"))[0]
    left_wrist_path: Path = list(day_path.glob("left wrist.csv"))[0]
    right_wrist_path: Path = list(day_path.glob("right wrist.csv"))[0]

    left_finger: pd.DataFrame = pd.read_csv(left_finger_path)
    right_finger: pd.DataFrame = pd.read_csv(right_finger_path)
    left_wrist: pd.DataFrame = pd.read_csv(left_wrist_path)
    right_wrist: pd.DataFrame = pd.read_csv(right_wrist_path)

    return {"left finger": left_finger,
            "right finger": right_finger,
            "left wrist": left_wrist,
            "right wrist": right_wrist}

In [ ]:
# Gets all subject directories inside the main data folder, ignoring hidden files
# For each subject folder:
    # Calls get_day_paths() to get all day paths.
    # Stores them in a dictionary where the key is the subject name and the value is a list of day paths.
def get_subject_day_paths(data_path: Path) -> dict[str, list[Path]]: # checked
    all_subject_dir_names: list[str] = os.listdir(data_path)
    all_subject_paths: list[Path] = sorted(list(
        map(lambda x: data_path / x, list(filter(lambda x: x[0] != ".", all_subject_dir_names)))
    ))
    subject_day_paths = {}

    for subject_path in all_subject_paths:
        subject = subject_path.name
        subject_day_paths[subject] = get_day_paths(subject_path=subject_path)

    return subject_day_paths

In [ ]:
# Prepares the output folder structure to store interpolated files.
def create_dirs(valid_subject_day_paths: dict[str, list[Path]], output_dir: Path): # checked
    subjects: list[str] = valid_subject_day_paths.keys()

    for subject in subjects:
        subject_output_path: Path = output_dir / subject
        if not os.path.exists(subject_output_path):
            os.makedirs(subject_output_path)

        day_paths: list[Path] = valid_subject_day_paths[subject]
        days: list[str] = list(map(lambda day_path: day_path.name[:6], day_paths))

        for day, day_path in zip(days, day_paths):
            day_output_path: Path = output_dir / subject / day
            if not os.path.exists(day_output_path):
                os.makedirs(day_output_path)

            files_in_day_path: list[str] = os.listdir(day_path)
            if "segment1" in files_in_day_path and not os.path.exists(day_output_path / "segment1"):
                os.makedirs(day_output_path / "segment1")

            if "segment2" in files_in_day_path and not os.path.exists(day_output_path / "segment2"):
                os.makedirs(day_output_path / "segment2")

            if "segment3" in files_in_day_path and not os.path.exists(day_output_path / "segment3"):
                os.makedirs(day_output_path / "segment3")

In [ ]:
# Checking for subjects with segments 
def check_interpolate_sensor_data(data_path: Path):
    # Calls get_subject_day_paths() to gather subject → days.
    valid_subject_day_paths: dict[str, list[Path]] = get_subject_day_paths(data_path=data_path)

    subjects: list[str] = valid_subject_day_paths.keys()
    for subject in subjects:
        day_paths = valid_subject_day_paths[subject]

        for day_path in day_paths:
            files_in_day_path: list[str] = os.listdir(day_path)

            # If the day folder does not contain segments
            if "segment1" not in files_in_day_path:
                continue
            elif "segment1" in files_in_day_path:
                print(subject, day_path.name[:6])

check_interpolate_sensor_data(data_path=Path("path/to/your/data"))

In [ ]:
# Converts a string timestamp (formatted as YYYY-MM-DD HH:MM:SS.ssssss) into a Python datetime object.
# Used later in interpolation so timestamps can be transformed into numeric values.
def convert_str_to_datetime(datetime_str: str) -> datetime:
    datetime_obj = datetime.strptime(datetime_str, '%Y-%m-%d %H:%M:%S.%f')
    return datetime_obj

In [ ]:
# Takes timestamp data (t) and sensor values (x) and resamples them at a consistent frequency 
# (sample_rate Hz, here 50 Hz)
def interpolate(t, x, sample_rate, axis=0, kind='linear', threshold=3): 
    t = np.array(list(map(lambda x: convert_str_to_datetime(x).timestamp(), t)))
    t0, tN = t[0], t[len(t)-1]
    dt = 1 / sample_rate
    t_ = np.arange(t0, tN+dt/2, dt).round(threshold)  # Round is to resolve floating point issues
    f = interp1d(t, x, axis=axis, kind=kind, copy=False, fill_value='extrapolate')
    return t_, f(t_)

In [ ]:
# Ensures all sensor data streams (fingers and wrists) align with the left wrist data 
# left wrist data (used as the reference timeline)
def synchronize(left_wrist_t, acc, loc): 
    data_len = len(left_wrist_t)
    # If the sensor location is not the left wrist, the accelerometer signal is resampled so it matches the length of left_wrist_t
    if loc != 'left wrist':
        acc = resample(acc, data_len)

    # Both signals are then interpolated to 50 Hz using interpolate().
    interp_time, interp_acc = interpolate(left_wrist_t.copy(), acc, sample_rate=50)
    return interp_time, interp_acc

In [ ]:
# Main driver function
def interpolate_sensor_data(data_path: Path, output_dir: Path):
    # Calls get_subject_day_paths() to gather subject → days.
    valid_subject_day_paths: dict[str, list[Path]] = get_subject_day_paths(data_path=data_path)
    # Calls create_dirs() so the destination directories exist.
    create_dirs(valid_subject_day_paths=valid_subject_day_paths, output_dir=output_dir)

    subjects: list[str] = valid_subject_day_paths.keys()
    for subject in subjects:
        day_paths = valid_subject_day_paths[subject]

        for day_path in day_paths:
            print(subject, day_path.name[:6])
            files_in_day_path: list[str] = os.listdir(day_path)

            # If the day folder does not contain segments
            if "segment1" not in files_in_day_path:
                # Reads trimmed CSVs for that day
                day_dict: dict[str, pd.DataFrame] = read_trimmed_sensor_csv(day_path)
                # Synchronizes finger/wrist signals to the left wrist
                synchronized_left_finger_time, synchronized_left_finger_acc = synchronize(left_wrist_t=day_dict["left wrist"]["timestamp"], acc=day_dict["left finger"]["magnitude"], loc="left finger")

                interpolated_day_dict = {
                    "left finger": synchronized_left_finger_acc,
                    "right finger": synchronize(left_wrist_t=day_dict["left wrist"]["timestamp"], acc=day_dict["right finger"]["magnitude"], loc="right finger")[1],
                    "left wrist": synchronize(left_wrist_t=day_dict["left wrist"]["timestamp"], acc=day_dict["left wrist"]["magnitude"], loc="left wrist")[1],
                    "right wrist": synchronize(left_wrist_t=day_dict["left wrist"]["timestamp"], acc=day_dict["right wrist"]["magnitude"], loc="right wrist")[1],
                }

                df_interpolated_day = pd.DataFrame({"timestamp": synchronized_left_finger_time,
                                                    "left finger": pd.Series(interpolated_day_dict["left finger"]),
                                                    "right finger": pd.Series(interpolated_day_dict["right finger"]),
                                                    "left wrist": pd.Series(interpolated_day_dict["left wrist"]),
                                                    "right wrist": pd.Series(interpolated_day_dict["right wrist"]),
                                                    })
                
                df_interpolated_day.to_csv(output_dir / subject / day_path.name / f"interpolated.csv", index=False)
            
            # If the day folder does contain segments 
            elif "segment1" in files_in_day_path:
                # Loops through each segment folder
                segment_paths: list[Path] = list(day_path.glob("segment*"))

                for segment_path in segment_paths:
                    # Reads CSVs for that segment
                    day_dict: dict[str, pd.DataFrame] = read_trimmed_sensor_csv(segment_path)
                    # Synchronizes all signals to left wrist
                    synchronized_left_finger_time, synchronized_left_finger_acc = synchronize(left_wrist_t=day_dict["left wrist"]["timestamp"], acc=day_dict["left finger"]["magnitude"], loc="left finger")

                    interpolated_day_dict = {
                        "left finger": synchronized_left_finger_acc,
                        "right finger": synchronize(left_wrist_t=day_dict["left wrist"]["timestamp"], acc=day_dict["right finger"]["magnitude"], loc="right finger")[1],
                        "left wrist": synchronize(left_wrist_t=day_dict["left wrist"]["timestamp"], acc=day_dict["left wrist"]["magnitude"], loc="left wrist")[1],
                        "right wrist": synchronize(left_wrist_t=day_dict["left wrist"]["timestamp"], acc=day_dict["right wrist"]["magnitude"], loc="right wrist")[1],
                    }

                    df_interpolated_day = pd.DataFrame({"timestamp": synchronized_left_finger_time,
                                                        "left finger": pd.Series(interpolated_day_dict["left finger"]),
                                                        "right finger": pd.Series(interpolated_day_dict["right finger"]),
                                                        "left wrist": pd.Series(interpolated_day_dict["left wrist"]),
                                                        "right wrist": pd.Series(interpolated_day_dict["right wrist"]),
                                                        })
                    
                    df_interpolated_day.to_csv(output_dir / subject / day_path.name / segment_path.name / f"interpolated.csv", index=False)


In [ ]:
interpolate_sensor_data(data_path=Path("path/to/your/data"),
                        output_dir=Path("path/to/your/data"))
